> **対応するブログ記事**: [#9 COSMICデータベースでがん関連タンパク質を特定する](../blog/article-09-cosmic.md)
>
> このNotebookはブログ記事 #9 のコードをセルごとに実行できるインタラクティブ版です。COSMICデータベースや集合演算の詳しい解説はブログ記事を参照してください。

# Step 9: COSMIC照合

COSMIC Cancer Gene Census（CGC）の遺伝子リストと、DIA-MSで同定されたタンパク質を照合し、カバー率を算出する。

In [ ]:
# データ操作ライブラリpandasをインポート
import pandas as pd
# グラフ描画ライブラリmatplotlibをインポート
import matplotlib.pyplot as plt
# ファイルパス操作用のPathlibをインポート
from pathlib import Path

# 結果ファイルを格納するディレクトリへのパス
RESULTS = Path("..") / "results"
# 図の保存先ディレクトリへのパス
FIG_DIR = RESULTS / "figures"
# テーブル（CSV等）の保存先ディレクトリへのパス
TABLE_DIR = RESULTS / "tables"

## COSMIC遺伝子リスト

In [ ]:
# 大腸がん（CRC）特異的 COSMIC Cancer Gene Census 遺伝子リスト（65遺伝子）
# COSMICはがん関連遺伝子のキュレーション済みデータベース
COSMIC_CRC_GENES = [
    # --- Wntシグナル経路 ---
    # 大腸がんで最も高頻度に変異する経路。APC変異は大腸がんの約80%で検出される
    "APC", "CTNNB1", "RNF43", "ZNRF3", "AXIN2", "AMER1", "SOX9", "TCF7L2", "DCC",
    # --- TP53 / 細胞周期制御 ---
    # がん抑制遺伝子TP53と細胞分裂を制御するサイクリン・CDK関連遺伝子
    "TP53", "RB1", "CDK4", "CDK8", "CCND1", "CHEK2", "RAD51",
    # --- RAS/MAPK経路 ---
    # 細胞増殖シグナルの中核経路。KRASは大腸がんの約40%で変異
    "KRAS", "NRAS", "BRAF",
    # --- PI3K/AKT経路 ---
    # 細胞生存・増殖を促進するシグナル経路
    "PIK3CA", "PTEN",
    # --- TGF-βシグナル経路 ---
    # 細胞増殖抑制・分化誘導に関わるシグナル経路
    "SMAD4", "SMAD2", "TGFBR2", "ACVR2A", "BMP4",
    # --- DNAミスマッチ修復（MMR）関連遺伝子 ---
    # DNA複製エラーを修復する機構。欠損するとマイクロサテライト不安定性（MSI）を引き起こす
    "MSH6", "MSH2", "MLH1", "PMS2", "MUTYH", "POLE", "POLD1",
    # --- DNA損傷応答（DDR）関連遺伝子 ---
    # DNA二本鎖切断の修復に関わる遺伝子群
    "ATM", "BRCA1", "BRCA2", "PALB2",
    # --- 受容体チロシンキナーゼ（RTK）---
    # 細胞外シグナルを細胞内に伝達する膜受容体。分子標的薬の標的となる
    "ERBB2", "ERBB3", "MET", "IGF1R", "PDGFRA",
    "FGFR1", "FGFR2", "FGFR3", "ALK", "ROS1", "RET",
    # --- E3ユビキチンリガーゼ ---
    # タンパク質分解を制御する酵素。FBXW7はがん抑制因子として機能
    "FBXW7",
    # --- エピジェネティクス関連遺伝子 ---
    # クロマチン構造やヒストン修飾を制御し、遺伝子発現を調節する
    "ARID1A", "CREBBP", "EP300", "KMT2A", "KMT2D",
    # --- JAK/STAT・NOTCH経路 ---
    # 免疫応答・細胞分化に関わるシグナル経路
    "JAK2", "STAT3", "NOTCH1", "NOTCH2",
    # --- 代謝・シグナル関連遺伝子 ---
    # GNASはGタンパク質、IDH1/2は代謝酵素で変異によりがん代謝を促進
    "GNAS", "IDH1", "IDH2",
    # --- 細胞接着関連遺伝子 ---
    # CDH1（E-カドヘリン）は細胞間接着に関与し、欠損で浸潤・転移を促進
    "CDH1",
    # --- 古典的がん抑制遺伝子 ---
    # 様々ながん種で変異が報告される代表的ながん抑制遺伝子
    "NF1", "VHL", "WT1",
]

# CRC特異的遺伝子リストの総数を表示して確認
print(f"CRC genes: {len(COSMIC_CRC_GENES)}")

In [ ]:
# 全がん種 COSMIC 遺伝子リスト = CRC特異的65遺伝子 + 追加約133遺伝子
# CRC以外のがん種（白血病、乳がん、肺がん等）で重要な遺伝子を追加
COSMIC_ALL_GENES = COSMIC_CRC_GENES + [
    # --- ABLファミリー（チロシンキナーゼ）---
    "ABL1", "ABL2",
    # --- ケモカイン受容体・転写因子 ---
    "ACKR3", "AFF4", "AKAP9",
    # --- AKTセリン/スレオニンキナーゼファミリー ---
    "AKT1", "AKT2",
    # --- 代謝酵素・構造タンパク質 ---
    "ALDH2", "ANK1",
    # --- APC関連・ホルモン受容体・RAFキナーゼ ---
    "APC2", "AR", "ARAF", "ARFRP1",
    # --- RhoGTPase関連遺伝子 ---
    "ARHGAP26", "ARHGEF12",
    # --- クロマチンリモデリング関連 ---
    "ARID2", "ARID5B",
    # --- ポリコーム関連・DNA損傷応答 ---
    "ASXL1", "ASXL2", "ATR", "ATRX",
    # --- MHCクラスI・がん抑制・転写因子 ---
    "B2M", "BAP1", "BCL2", "BCL6", "BCOR", "BCORL1", "BCR",
    # --- アポトーシス制御・DNA修復・受容体 ---
    "BIRC3", "BLM", "BMPR1A", "BTK", "BUB1B",
    # --- カルシウムチャネル・JAK-STAT関連 ---
    "CACNA1D", "CALR", "CARD11", "CARS1", "CASP8",
    # --- 転写因子・ユビキチンリガーゼ ---
    "CBFB", "CBL", "CCNB1IP1",
    # --- サイクリンDファミリー・サイクリンE ---
    "CCND2", "CCND3", "CCNE1",
    # --- 免疫チェックポイント・B細胞受容体 ---
    "CD274", "CD79A", "CD79B", "CDC73",
    # --- カドヘリン・CDK関連 ---
    "CDH11", "CDK12", "CDK6",
    # --- CDK阻害因子（がん抑制遺伝子）---
    "CDKN1A", "CDKN1B", "CDKN2A", "CDKN2B", "CDKN2C",
    # --- 転写因子・クロマチンリモデリング・細胞周期チェックポイント ---
    "CEBPA", "CHD4", "CHEK1", "CIC", "CIITA",
    # --- 細胞周期・代謝・細胞外マトリックス ---
    "CKS1B", "CMPK1", "COL1A1", "CREB1", "CREB3L2", "CRLF2",
    # --- コロニー刺激因子受容体・転写因子 ---
    "CSF1R", "CSF3R", "CUX1", "CYLD", "DAXX", "DDR2", "DDX3X",
    # --- RNA処理関連・DNAメチル化 ---
    "DICER1", "DNMT3A", "DROSHA",
    # --- 上皮成長因子受容体・翻訳開始因子・転写因子 ---
    "EGFR", "EIF4A2", "ELF4", "ELL", "EP400",
    # --- エフリン受容体・ErbBファミリー ---
    "EPHA3", "EPHA7", "EPHB1", "ERBB4", "ERG", "ESR1",
    # --- ETSファミリー転写因子 ---
    "ETV1", "ETV4", "ETV5", "ETV6",
    # --- RNA結合タンパク質・ヘパラン硫酸関連・ヒストン修飾 ---
    "EWSR1", "EXT1", "EXT2", "EZH2", "FAM46C",
    # --- ファンコニ貧血関連遺伝子群（DNA修復） ---
    "FANCA", "FANCC", "FANCD2", "FANCE", "FANCF", "FANCG",
    # --- アポトーシス受容体・カドヘリンスーパーファミリー ---
    "FAS", "FAT1", "FAT4",
    # --- FGFリガンドファミリー ---
    "FGF19", "FGF3", "FGF4",
    # --- 代謝酵素・がん抑制・転写因子 ---
    "FH", "FLCN", "FLI1",
    # --- VEGF/FLT受容体チロシンキナーゼファミリー ---
    "FLT1", "FLT3", "FLT4",
    # --- フォークヘッド転写因子ファミリー ---
    "FOXA1", "FOXL2", "FOXO1", "FOXP1", "FUBP1", "FUS",
    # --- 成長因子・GATA転写因子ファミリー ---
    "GAS7", "GATA1", "GATA2", "GATA3",
    # --- Gタンパク質αサブユニット・プロテオグリカン ---
    "GNA11", "GNA13", "GNAQ", "GPC3", "GRM3",
    # --- ヒストンH3バリアント ---
    "H3-3A", "H3-3B", "H3C2",
    # --- 肝細胞増殖因子・ヒストン ---
    "HGF", "HIST1H3B",
]

# 全がん種遺伝子リストの総数を表示して確認（CRC 65 + 追加遺伝子）
print(f"All-cancer genes: {len(COSMIC_ALL_GENES)}")

## 照合

In [ ]:
# 前処理済みデータを読み込み（行インデックス=タンパク質名/遺伝子名）
df = pd.read_csv(RESULTS / "preprocessed_data.csv", index_col=0)
# DIA-MSで同定されたタンパク質名をset（集合）に変換（高速な集合演算のため）
identified = set(df.index)
# 同定タンパク質の総数を表示
print(f"同定タンパク質数: {len(identified)}")

# COSMIC遺伝子リストもset（集合）に変換して集合演算に備える
cosmic_all = set(COSMIC_ALL_GENES)  # 全がん種COSMIC遺伝子の集合
cosmic_crc = set(COSMIC_CRC_GENES)  # CRC特異的COSMIC遺伝子の集合

# &演算子で積集合（intersection）を計算：両方に含まれる遺伝子 = 実際に検出されたがん関連タンパク質
overlap_all = identified & cosmic_all  # 同定タンパク質 ∩ 全がん種COSMIC遺伝子
overlap_crc = identified & cosmic_crc  # 同定タンパク質 ∩ CRC特異的COSMIC遺伝子

# カバー率 = (検出されたCOSMIC遺伝子数 / COSMIC遺伝子総数) × 100 [%]
cov_all = len(overlap_all) / len(cosmic_all) * 100  # 全がん種のカバー率
cov_crc = len(overlap_crc) / len(cosmic_crc) * 100  # CRC特異的のカバー率

# 照合結果をフォーマット付きで表示
print(f"\n全がん関連: {len(overlap_all)}/{len(cosmic_all)} ({cov_all:.1f}%)")
print(f"CRC関連:    {len(overlap_crc)}/{len(cosmic_crc)} ({cov_crc:.1f}%)")

## 可視化

In [ ]:
# 1行2列のサブプロット領域を作成（横12インチ×縦5インチ）
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# 棒グラフの色を定義：オレンジ=COSMICの全数、緑=今回検出された数
colors = ["#FFB74D", "#4CAF50"]

# 各パネルの設定をリストにまとめる（軸, タイトル, COSMIC遺伝子数, 重複数, カバー率, ラベル）
panels = [
    (axes[0], "Cancer-Associated Proteins", len(cosmic_all), len(overlap_all), cov_all, "All Cancer"),
    (axes[1], "CRC-Associated Proteins",    len(cosmic_crc), len(overlap_crc), cov_crc, "CRC"),
]

# 各パネルについてループで棒グラフを描画
for ax, title, n_cosmic, n_overlap, cov, label in panels:
    # x軸のカテゴリラベル（COSMICの全数 vs 本研究で検出された数）
    cats = [f"COSMIC\n({label})", "Identified\nin This Study"]
    # y軸の値（COSMIC遺伝子総数と重複遺伝子数）
    vals = [n_cosmic, n_overlap]
    # 棒グラフを描画（幅0.5、白い枠線で区切り）
    bars = ax.bar(cats, vals, color=colors, width=0.5, edgecolor="white")
    # グラフタイトルにカバー率を含めて表示
    ax.set_title(f"{title}\n(Coverage: {cov:.1f}%)")
    # y軸ラベルを設定
    ax.set_ylabel("Number of Proteins")
    # 各棒の上に数値ラベルを表示（棒の上端から+2の位置、中央揃え、太字）
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                str(v), ha="center", fontweight="bold")
    # 上辺の枠線を非表示にして見た目をすっきりさせる
    ax.spines["top"].set_visible(False)
    # 右辺の枠線を非表示にして見た目をすっきりさせる
    ax.spines["right"].set_visible(False)

# サブプロット間の余白を自動調整
plt.tight_layout()
# 図をPNG形式で保存（解像度150dpi、余白を最小化）
fig.savefig(FIG_DIR / "fig_cosmic_coverage.png", dpi=150, bbox_inches="tight")
# 図を画面に表示
plt.show()

In [ ]:
# --- 照合結果をCSVファイルとして保存 ---
# 全がん種で重複した遺伝子をDataFrameに変換（アルファベット順にソート、Type列で分類）
overlap_df = pd.DataFrame({"Gene": sorted(overlap_all), "Type": "All_Cancer"})
# CRC特異的に重複した遺伝子をDataFrameに変換
crc_df = pd.DataFrame({"Gene": sorted(overlap_crc), "Type": "CRC_Specific"})
# 2つのDataFrameを縦方向に結合（全がん種 + CRC特異的）
out = pd.concat([overlap_df, crc_df])
# 結合したDataFrameをCSVとして保存（インデックス列は不要なので除外）
out.to_csv(TABLE_DIR / "cosmic_overlap.csv", index=False)
# 保存先パスと行数を表示して確認
print(f"保存: {TABLE_DIR / 'cosmic_overlap.csv'} ({len(out)} rows)")